1\. Displays Customers with their latest order date

In [4]:
SELECT c.CustomerID, c.CustomerName, 
       (SELECT MAX(OrderDate) 
        FROM Sales.Orders o 
        WHERE o.CustomerID = c.CustomerID) AS LastOrderDate
FROM Sales.Customers c;


(663 rows affected)

Total execution time: 00:00:00.088

CustomerID,CustomerName,LastOrderDate
1,Tailspin Toys (Head Office),2016-05-27
2,"Tailspin Toys (Sylvanite, MT)",2016-05-14
3,"Tailspin Toys (Peeples Valley, AZ)",2016-05-30
4,"Tailspin Toys (Medicine Lodge, KS)",2016-04-28
5,"Tailspin Toys (Gasport, NY)",2016-05-28
6,"Tailspin Toys (Jessie, ND)",2016-05-31
7,"Tailspin Toys (Frankewing, TN)",2016-05-27
8,"Tailspin Toys (Bow Mar, CO)",2016-05-20
9,"Tailspin Toys (Netcong, NJ)",2016-05-24
10,"Tailspin Toys (Wimbledon, ND)",2016-05-24


2\. Customers who placed multiple orders on the same date

In [49]:
SELECT CustomerID, OrderDate, COUNT(OrderID) AS OrderCount
FROM Sales.Orders
GROUP BY CustomerID, OrderDate
HAVING COUNT(OrderID) > 1;


(9894 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.102

CustomerID,OrderDate,OrderCount
1004,2015-02-25,2
43,2015-08-25,2
428,2015-06-09,2
581,2014-03-18,2
172,2015-01-03,2
521,2015-03-04,3
118,2015-10-20,2
1007,2015-02-17,2
171,2015-08-19,2
414,2016-02-19,2


3\. Suppliers that supplied products that are actually in orders (this way you know which suppliers are actually selling)

In [17]:
SELECT SupplierID, SupplierName
FROM Purchasing.Suppliers
WHERE SupplierID IN (
    SELECT DISTINCT SupplierID 
    FROM Warehouse.StockItems 
    WHERE StockItemID IN (SELECT DISTINCT StockItemID FROM Sales.OrderLines)
);

(7 rows affected)

Total execution time: 00:00:00.090

SupplierID,SupplierName
1,A Datum Corporation
2,"Contoso, Ltd."
4,"Fabrikam, Inc."
5,Graphic Design Institute
7,"Litware, Inc."
10,Northwind Electric Cars
12,The Phone Company


4. HIghest Sellers

In [51]:
SELECT o.SalespersonPersonID, p.FullName, SUM(ol.Quantity * ol.UnitPrice) AS TotalSales
FROM Sales.Orders o
INNER JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
INNER JOIN Application.People p ON o.SalespersonPersonID = p.PersonID
GROUP BY o.SalespersonPersonID, p.FullName
ORDER BY TotalSales DESC;


(10 rows affected)

Total execution time: 00:00:00.086

SalespersonPersonID,FullName,TotalSales
16,Archer Lamble,18551146.95
2,Kayla Woodcock,18107095.00
3,Hudson Onslow,17815605.10
15,Taj Shand,17812364.60
6,Sophia Hinton,17768199.25
13,Hudson Hollinworth,17716354.25
20,Jack Potter,17621145.20
14,Lily Code,17612639.80
7,Amy Trefl,17329344.05
8,Anthony Grosse,17300382.20


5. Most Recent orders for each customer

In [39]:
SELECT o.CustomerID, o.OrderID, o.OrderDate
FROM Sales.Orders o
INNER JOIN (
    SELECT CustomerID, MAX(OrderDate) AS LatestOrder
    FROM Sales.Orders
    GROUP BY CustomerID
) latest ON o.CustomerID = latest.CustomerID AND o.OrderDate = latest.LatestOrder
ORDER BY o.CustomerID;

(785 rows affected)

Total execution time: 00:00:00.075

CustomerID,OrderID,OrderDate
1,73290,2016-05-27
2,72475,2016-05-14
3,73465,2016-05-30
4,71366,2016-04-28
5,73407,2016-05-28
6,73547,2016-05-31
7,73365,2016-05-27
7,73393,2016-05-27
8,72847,2016-05-20
9,73059,2016-05-24


6. Top 5 selling products

In [38]:
WITH ProductOrders AS (
    SELECT StockItemID, COUNT(*) AS OrderCount
    FROM Sales.OrderLines
    GROUP BY StockItemID
)
SELECT TOP 5 si.StockItemID, si.StockItemName, po.OrderCount
FROM ProductOrders po
INNER JOIN Warehouse.StockItems si ON po.StockItemID = si.StockItemID
ORDER BY po.OrderCount DESC



(5 rows affected)

Total execution time: 00:00:00.019

StockItemID,StockItemName,OrderCount
104,Alien officer hoodie (Black) 3XL,1123
120,Dinosaur battery-powered slippers (Green) L,1123
167,10 mm Anti static bubble wrap (Blue) 50m,1119
13,USB food flash drive - shrimp cocktail,1117
88,"""The Gu"" red shirt XML tag t-shirt (White) 7XL",1110


7. Total Revenue per customer

In [37]:
WITH CustomerRevenue AS (
    SELECT o.CustomerID, SUM(ol.Quantity * ol.UnitPrice) AS TotalRevenue
    FROM Sales.Orders o
    INNER JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
    GROUP BY o.CustomerID
)
SELECT c.CustomerID, c.CustomerName, cr.TotalRevenue
FROM Sales.Customers c
INNER JOIN CustomerRevenue cr ON c.CustomerID = cr.CustomerID
ORDER BY cr.TotalRevenue DESC;


(663 rows affected)

Total execution time: 00:00:00.069

CustomerID,CustomerName,TotalRevenue
149,"Tailspin Toys (Inguadona, MN)",384393.35
132,"Tailspin Toys (Minidoka, ID)",379660.70
977,Mauno Laurila,377189.80
580,"Wingtip Toys (Sarversville, PA)",372350.00
964,Ingrida Zeltina,368067.45
14,"Tailspin Toys (Long Meadow, MD)",367258.50
954,Nasrin Omidzadeh,366883.75
593,"Wingtip Toys (Cuyamungue, NM)",365915.45
472,"Wingtip Toys (San Jacinto, CA)",365330.95
550,"Wingtip Toys (Morrison Bluff, AR)",360652.80


8. Total Transactions per supplier

In [41]:
SELECT s.SupplierID, s.SupplierName, t.TotalTransactions
FROM Purchasing.Suppliers s
INNER JOIN (
    SELECT SupplierID, COUNT(*) AS TotalTransactions
    FROM Purchasing.SupplierTransactions
    GROUP BY SupplierID
) t ON s.SupplierID = t.SupplierID
ORDER BY t.TotalTransactions DESC;


(7 rows affected)

Total execution time: 00:00:00.013

SupplierID,SupplierName,TotalTransactions
4,"Fabrikam, Inc.",1232
7,"Litware, Inc.",1160
5,Graphic Design Institute,16
10,Northwind Electric Cars,14
12,The Phone Company,7
1,A Datum Corporation,7
2,"Contoso, Ltd.",2


9. Customer Categories and the number of orders they have. ordered by greatest

In [44]:
SELECT cc.CustomerCategoryName, COUNT(o.OrderID) AS TotalOrders
FROM Sales.Orders o
INNER JOIN Sales.Customers c 
ON o.CustomerID = c.CustomerID
INNER JOIN Sales.CustomerCategories cc ON c.CustomerCategoryID = cc.CustomerCategoryID
GROUP BY cc.CustomerCategoryName
ORDER BY TotalOrders DESC;


(5 rows affected)

Total execution time: 00:00:00.072

CustomerCategoryName,TotalOrders
Novelty Shop,52547
Supermarket,6022
Gift Store,5089
Computer Store,5041
Corporate,4896


10. Employees who haven't placed any orders for customers (rip)

In [46]:
SELECT e.PersonID, e.FullName
FROM Application.People e
WHERE e.IsEmployee = 1
AND NOT EXISTS (
    SELECT 1 FROM Sales.Orders o
    WHERE o.SalespersonPersonID = e.PersonID
);


(9 rows affected)

Total execution time: 00:00:00.044

PersonID,FullName
9,Alica Fatnowna
11,Ethan Onslow
5,Eva Muirden
12,Henry Forlonge
4,Isabella Rupp
19,Jai Shand
18,Katie Darwin
17,Piper Koch
10,Stella Rosenhain
